# Lab 2: Agents — Constructing a Goal-Based Agent using a Large Language Model

## Objective

This laboratory focuses on constructing a goal-based intelligent agent for a warehouse navigation problem using a Large Language Model (LLM) as a software engineering assistant.

The laboratory covers:

* Understanding the warehouse navigation problem
* Designing a goal-based agent
* Using an LLM to generate Python code
* Executing and testing the generated program
* Evaluating the selected search algorithm
* Improving the program through iterative prompting when necessary


## Task 1 — Understanding the Problem

### 1. What is the environment?

The environment is a two-dimensional grid representing a warehouse. It contains free cells, obstacles representing shelving units, a starting position `S`, and a goal position `G`.

### 2. What is the goal of the agent?

The goal is to determine a collision-free path that takes the warehouse vehicle from the starting position `S` to the destination `G`.

### 3. What actions are available to the agent?

The vehicle can move one grid square at a time in four directions:

* Up
* Down
* Left
* Right

The vehicle cannot move through obstacles or outside the warehouse grid.

### 4. What information must the agent maintain in order to choose its next action?

The agent needs to maintain:

* Its current position on the grid
* The warehouse layout
* The position of the goal
* The positions of obstacles
* The possible valid neighboring cells

This information allows the agent to determine which movements are possible and which actions can help it reach the goal.

### 5. Why is this a goal-based agent rather than a simple reflex agent?

This is a goal-based agent because it has an explicit objective: reaching the destination `G`. The agent must consider the current state of the warehouse and select actions that help it achieve this goal while avoiding obstacles.

A simple reflex agent would mainly select an action based on the current situation, whereas a goal-based agent considers whether its actions contribute toward achieving a specified goal.

### Think About It — Larger Warehouse

If the warehouse becomes twice as large, the same general search strategy could still be used, but the number of possible states and paths would increase. As a result, the search may require more time and memory, and choosing an efficient search strategy could become more important.


## Task 2 — Designing the Agent

### Agent Design

| Component                     | Description                                                                                                                    |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------ |
| **Environment**               | The warehouse represented as a two-dimensional grid containing free cells, obstacles, a starting position `S`, and a goal `G`. |
| **Current State**             | The current row and column position of the vehicle in the warehouse grid.                                                      |
| **Goal**                      | Reach the destination `G` from the starting position `S` without passing through obstacles.                                    |
| **Available Actions**         | Move Up, Down, Left, or Right by one grid square.                                                                              |
| **Decision-Making Component** | A search algorithm that explores possible movements and selects a collision-free sequence of actions leading from `S` to `G`.  |

### Goal-Based Agent Architecture

```text
                  ┌─────────────────────┐
                  │     Environment     │
                  │    Warehouse Grid   │
                  └──────────┬──────────┘
                             │
                             ▼
                  ┌─────────────────────┐
                  │    Current State    │
                  │   Vehicle Position  │
                  └──────────┬──────────┘
                             │
                             ▼
                  ┌─────────────────────┐
                  │  Decision-Making    │
                  │   Search Algorithm  │
                  └──────────┬──────────┘
                             │
                             ▼
                  ┌─────────────────────┐
                  │       Action        │
                  │   ↑  ↓  ←  →       │
                  └──────────┬──────────┘
                             │
                             ▼
                  ┌─────────────────────┐
                  │     New State       │
                  │   Vehicle Position  │
                  └──────────┬──────────┘
                             │
                             └───────────────┐
                                             │
                                             ▼
                                    Repeat until G
```

The agent observes its current state in the warehouse, uses its decision-making component to select an action, performs that action, and reaches a new state. This process continues until the goal state `G` is reached.


## Task 3 — Prompt Engineering

Before generating the program, the warehouse navigation problem was specified clearly to the LLM.

### Prompt Used

> Write a well-documented Python program implementing a goal-based agent for the following warehouse navigation problem.
>
> Warehouse map:
>
> #####################
> #S....#............G#
> #.##....##########..#
> #....##.............#
> #.######.###.#.###..#
> #........#..........#
> #####################
>
> Where:
>
> * `S` is the starting position.
> * `G` is the goal position.
> * `#` represents an obstacle that cannot be crossed.
> * `.` represents a free cell.
>
> The vehicle can move one grid square at a time in four directions:
> Up, Down, Left, and Right.
>
> Requirements:
>
> 1. Represent the warehouse as a two-dimensional grid.
> 2. Locate `S` and `G` from the grid.
> 3. Determine a collision-free path from `S` to `G`.
> 4. The agent must not move outside the grid or through obstacles.
> 5. Print the path found as a sequence of positions or moves.
> 6. Print a suitable message if no path exists.
> 7. Explain the search algorithm used and why it is appropriate for this goal-based navigation problem.
> 8. Keep the implementation simple and suitable for a laboratory exercise.
> 9. Include enough comments to make the agent's operation clear.
> 10. Do not use external libraries beyond Python's standard library.

### Testing the Generated Program

The generated program will be executed to determine whether it works correctly on the first attempt. Its path, obstacle avoidance, and ability to reach the goal will then be checked.


In [1]:
from collections import deque

warehouse = [
    list("#####################"),
    list("#S....#............G#"),
    list("#.##....##########..#"),
    list("#....##.............#"),
    list("#.######.###.#.###..#"),
    list("#........#..........#"),
    list("#####################")
]

MOVEMENTS = [
    (-1, 0),
    (1, 0),
    (0, -1),
    (0, 1)
]


def find_position(grid, symbol):
    for row in range(len(grid)):
        for col in range(len(grid[row])):
            if grid[row][col] == symbol:
                return (row, col)
    return None


def is_valid(grid, row, col):
    return (
        0 <= row < len(grid)
        and 0 <= col < len(grid[row])
        and grid[row][col] != '#'
    )


def bfs(grid, start, goal):

    queue = deque([start])
    visited = {start}
    parent = {start: None}

    while queue:
        current = queue.popleft()

        if current == goal:
            break

        row, col = current

        for dr, dc in MOVEMENTS:
            new_row = row + dr
            new_col = col + dc
            new_position = (new_row, new_col)

            if (
                is_valid(grid, new_row, new_col)
                and new_position not in visited
            ):
                visited.add(new_position)
                parent[new_position] = current
                queue.append(new_position)

    if goal not in parent:
        return None
    path = []
    current = goal

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()
    return path


def display_path(grid, path):
    display = [row[:] for row in grid]

    for row, col in path:
        if display[row][col] not in ('S', 'G'):
            display[row][col] = '*'

    for row in display:
        print(''.join(row))

start = find_position(warehouse, 'S')
goal = find_position(warehouse, 'G')

print("Start:", start)
print("Goal:", goal)

path = bfs(warehouse, start, goal)

if path is None:
    print("No collision-free path exists from S to G.")
else:
    print("\nCollision-free path found!")
    print("Path:", path)
    print("Number of moves:", len(path) - 1)

    print("\nWarehouse with path:")
    display_path(warehouse, path)

Start: (1, 1)
Goal: (1, 19)

Collision-free path found!
Path: [(1, 1), (1, 2), (1, 3), (1, 4), (2, 4), (2, 5), (2, 6), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (1, 16), (1, 17), (1, 18), (1, 19)]
Number of moves: 20

Warehouse with path:
#####################
#S***.#************G#
#.##****##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################


### First Attempt Results

The LLM-generated program worked correctly on the first attempt.

The program identified the starting position as `(1, 1)` and the goal position as `(1, 19)`. It then found a collision-free path between them.

**Result:**

* Search algorithm: Breadth-First Search (BFS)
* Path found: Yes
* Number of moves: 20
* Goal reached: Yes
* Obstacles crossed: No

The generated program also displayed the warehouse with the discovered path marked using `*`.

### Search Algorithm Selected

The LLM selected **Breadth-First Search (BFS)**. BFS uses a queue to explore the reachable grid positions level by level. Each movement has the same cost of one grid square, so BFS is appropriate for finding a shortest path in terms of the number of moves.

The path returned by the program was:

`(1,1) → (1,2) → (1,3) → (1,4) → (2,4) → (2,5) → (2,6) → (2,7) → (1,7) → ... → (1,19)`

The program therefore successfully demonstrated a goal-based agent that searches for a sequence of actions leading from the initial state to the specified goal while avoiding obstacles.


In [2]:
# Validate that the generated path is correct

def validate_path(grid, path, start, goal):
    if path is None:
        return False, "No path was generated."

    if path[0] != start:
        return False, "Path does not start at S."

    if path[-1] != goal:
        return False, "Path does not end at G."

    for i, (row, col) in enumerate(path):

        if not (0 <= row < len(grid) and 0 <= col < len(grid[row])):
            return False, f"Position {path[i]} is outside the grid."

        if grid[row][col] == '#':
            return False, f"Path crosses an obstacle at {path[i]}."

        if i > 0:
            previous_row, previous_col = path[i - 1]

            row_difference = abs(row - previous_row)
            col_difference = abs(col - previous_col)

            if row_difference + col_difference != 1:
                return False, f"Invalid movement from {path[i-1]} to {path[i]}."

    return True, "Path is valid."

valid, message = validate_path(warehouse, path, start, goal)

print("Path Validation:", message)

Path Validation: Path is valid.


### Evaluation of the Generated Program

The first LLM-generated program worked correctly without requiring a second prompt. It successfully identified the start and goal positions and generated a collision-free path from `S` to `G`.

The path was then independently validated using a separate validation function. The validation confirmed that the path:
- starts at `S`;
- ends at `G`;
- remains within the warehouse grid;
- does not pass through obstacles; and
- consists only of valid one-cell movements.

The program found a path of **20 moves**.

### Search Algorithm

The generated program uses **Breadth-First Search (BFS)**. BFS explores the reachable positions level by level using a queue. Since every movement has the same cost of one grid square, BFS is appropriate for this problem because it finds a shortest path in terms of the number of moves.

### Result

The first attempt was successful, so no second prompting iteration was required. This demonstrates that a clear specification of the environment, allowed actions, constraints, and expected output was sufficient for the LLM to generate a working goal-based agent.

The validation step was important because the generated code should be tested rather than assumed to be correct.